# snowloader

>[snowloader](https://github.com/ronidas39/snowloader) is a comprehensive ServiceNow data loader for AI/LLM pipelines. It pulls data from six core ServiceNow tables — Incidents, Knowledge Base, CMDB, Changes, Problems, and Service Catalog — and converts them into LangChain Documents.
>
>Built for production use with 4 authentication modes, retry logic with exponential backoff, rate limiting, delta sync, CMDB relationship traversal, and memory-efficient streaming.

## Overview

I built snowloader because I needed a reliable way to pull structured ServiceNow data into RAG pipelines. Existing tools either covered a single table, ignored relationships between configuration items, or required too much boilerplate.

### Features

| Feature | Description |
|---------|-------------|
| **6 loaders** | Incidents, KB articles, CMDB CIs, Changes, Problems, Catalog items |
| **CMDB relationships** | Concurrent graph traversal of CI dependencies |
| **Delta sync** | Only fetch records updated since your last sync |
| **HTML cleaning** | Built-in KB article HTML stripping, no extra dependencies |
| **Journal support** | Optionally include work notes and comments |
| **4 auth modes** | Basic, OAuth Password, OAuth Client Credentials, Bearer Token |
| **Production-grade** | Retry with backoff, rate limiting, thread safety, proxy support |

### Integration details

| Class | Package | Serializable | JS support |
|-------|---------|:---:|:---:|
| ServiceNowIncidentLoader | snowloader | beta | ❌ |
| ServiceNowKBLoader | snowloader | beta | ❌ |
| ServiceNowCMDBLoader | snowloader | beta | ❌ |
| ServiceNowChangeLoader | snowloader | beta | ❌ |
| ServiceNowProblemLoader | snowloader | beta | ❌ |
| ServiceNowCatalogLoader | snowloader | beta | ❌ |

## Setup

Install the package with the LangChain extra:

In [ ]:
%pip install -qU snowloader[langchain]

You need a ServiceNow instance with REST API access and a user account with appropriate roles (e.g., `itil` for incidents, `knowledge` for KB articles).

## Initialization

Create a connection to your ServiceNow instance:

In [ ]:
from snowloader import SnowConnection
from snowloader.adapters.langchain import ServiceNowIncidentLoader

conn = SnowConnection(
    instance_url="https://mycompany.service-now.com",
    username="admin",
    password="password",
)

For production, use OAuth Client Credentials (no user password needed):

```python
conn = SnowConnection(
    instance_url="https://mycompany.service-now.com",
    client_id="your_client_id",
    client_secret="your_client_secret",
)
```

## Load incidents

In [ ]:
loader = ServiceNowIncidentLoader(connection=conn, query="active=true^priority<=2")
docs = loader.load()

print(f"Loaded {len(docs)} incidents")
print(docs[0].page_content[:300])

In [ ]:
print(docs[0].metadata)

## Lazy loading (streaming)

For large tables, use `lazy_load()` to stream documents one at a time without holding everything in memory:

In [ ]:
for doc in loader.lazy_load():
    print(f"[{doc.metadata['number']}] {doc.page_content[:100]}")

## All available loaders

```python
from snowloader.adapters.langchain import (
    ServiceNowIncidentLoader,    # IT incidents
    ServiceNowKBLoader,          # Knowledge Base articles (HTML auto-cleaned)
    ServiceNowCMDBLoader,        # CMDB Configuration Items + relationships
    ServiceNowChangeLoader,      # Change requests
    ServiceNowProblemLoader,     # Problem records
    ServiceNowCatalogLoader,     # Service catalog items
)
```

## CMDB with relationship traversal

The CMDB loader can map out how configuration items relate to each other:

In [ ]:
from snowloader.adapters.langchain import ServiceNowCMDBLoader

cmdb_loader = ServiceNowCMDBLoader(
    connection=conn,
    ci_class="cmdb_ci_server",
    include_relationships=True,
)

server_docs = cmdb_loader.load()
print(server_docs[0].page_content)

## Delta sync

Only fetch records updated since your last sync:

In [ ]:
from datetime import datetime, timezone

# Only get incidents updated in the last 24 hours
since = datetime(2024, 6, 1, tzinfo=timezone.utc)
updated_docs = loader.load_since(since)
print(f"{len(updated_docs)} incidents updated since {since}")

## Use with a vector store

```python
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

docs = ServiceNowIncidentLoader(connection=conn, query="active=true").load()
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
results = retriever.invoke("network outage incidents")
```

## API reference

For a full list of parameters and configuration options, see the [snowloader documentation](https://snowloader.readthedocs.io).